In [13]:
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.


In [14]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# CIFAR-10 dataset statistics (mean and std for each RGB channel)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Define transformations for data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)  # Normalize with CIFAR-10 statistics
])

# Download and load the CIFAR-10 training data
trainset = datasets.CIFAR10(root='./data', download=True, train=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

In [15]:
# CIFAR-10 dataset statistics
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),   # Randomly flip images horizontally
    transforms.RandomCrop(32, padding=4),# Randomly crop with padding
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)  # Normalize with CIFAR-10 statistics
])

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(32*32*3, 256)
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(x.size(0), -1) # Flatten the image
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Initialize the model, loss function, and optimizer
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
for epoch in range(10): # 10 epochs
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()       # Zero the gradients
        
        outputs = model(images)     # Forward pass
        loss = criterion(outputs, labels) # Compute loss
        loss.backward()             # Backward pass
        optimizer.step()            # Update weights
        
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")


Epoch 1, Loss: 1.751060307178351
Epoch 2, Loss: 1.5496848885665464
Epoch 3, Loss: 1.4642588397121186
Epoch 4, Loss: 1.4012810573401049
Epoch 5, Loss: 1.3484241790173914
Epoch 6, Loss: 1.3048162592951293
Epoch 7, Loss: 1.264569067055612
Epoch 8, Loss: 1.227977576082015
Epoch 9, Loss: 1.1937533559091866
Epoch 10, Loss: 1.1613020921302268


In [17]:
# CIFAR-10 dataset statistics
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Validation data transformations (only normalization, no augmentation)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)  # Normalize with CIFAR-10 statistics
])

# Download and load the CIFAR-10 validation data
valset = datasets.CIFAR10(root='./data', download=True, train=False, transform=val_transform)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

In [18]:
# Validation loop
val_loss = 0.0
correct = 0
total = 0

# Switch the model to evaluation mode
model.eval()

with torch.no_grad():  # No need to track gradients during validation
    for images, labels in valloader:
        outputs = model(images)
        loss = criterion(outputs, labels)  # Compute the loss on the validation set
        val_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)  # Get the predicted class with the highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate average loss and accuracy
avg_val_loss = val_loss / len(valloader)
accuracy = 100 * correct / total

print(f"Validation Loss: {avg_val_loss}, Validation Accuracy: {accuracy}%")


Validation Loss: 1.3776774892381802, Validation Accuracy: 51.18%


In [19]:
import torch.nn.functional as F

class SimpleNNWithDropout(nn.Module):
    def __init__(self):
        super(SimpleNNWithDropout, self).__init__()
        self.fc1 = nn.Linear(32*32*3, 256)
        self.dropout = nn.Dropout(0.5)  # Drop 50% of neurons
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)  # Apply dropout
        return self.fc2(x)


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim



# Initialize the model, loss function, and optimizer
model = SimpleNNWithDropout()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
for epoch in range(10): # 10 epochs
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()       # Zero the gradients
        
        outputs = model(images)     # Forward pass
        loss = criterion(outputs, labels) # Compute loss
        loss.backward()             # Backward pass
        optimizer.step()            # Update weights
        
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")


Epoch 1, Loss: 1.8257546386755337
Epoch 2, Loss: 1.664789259891071
Epoch 3, Loss: 1.606063430266612
Epoch 4, Loss: 1.562518978972569
Epoch 5, Loss: 1.5324330023487511
Epoch 6, Loss: 1.511392644223045
Epoch 7, Loss: 1.489856301975982
Epoch 8, Loss: 1.468247650376976
Epoch 9, Loss: 1.4511706653763266
Epoch 10, Loss: 1.437798818282764


In [21]:
# Validation loop
val_loss = 0.0
correct = 0
total = 0

# Switch the model to evaluation mode
model.eval()

with torch.no_grad():  # No need to track gradients during validation
    for images, labels in valloader:
        outputs = model(images)
        loss = criterion(outputs, labels)  # Compute the loss on the validation set
        val_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)  # Get the predicted class with the highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate average loss and accuracy
avg_val_loss = val_loss / len(valloader)
accuracy = 100 * correct / total

print(f"Validation Loss: {avg_val_loss}, Validation Accuracy: {accuracy}%")

Validation Loss: 1.3978801373463527, Validation Accuracy: 51.05%


In [22]:
class SimpleNNWithBatchNorm(nn.Module):
    def __init__(self):
        super(SimpleNNWithBatchNorm, self).__init__()
        self.fc1 = nn.Linear(32*32*3, 256)
        self.batch_norm = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.batch_norm(self.fc1(x)))  # Apply batch normalization
        return self.fc2(x)


In [23]:
import torch
import torch.nn as nn
import torch.optim as optim



# Initialize the model, loss function, and optimizer
model = SimpleNNWithBatchNorm()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
for epoch in range(10): # 10 epochs
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()       # Zero the gradients
        
        outputs = model(images)     # Forward pass
        loss = criterion(outputs, labels) # Compute loss
        loss.backward()             # Backward pass
        optimizer.step()            # Update weights
        
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")


Epoch 1, Loss: 1.707577443946048
Epoch 2, Loss: 1.529111155311165
Epoch 3, Loss: 1.4469968809191223
Epoch 4, Loss: 1.3892426798715616
Epoch 5, Loss: 1.3381542846979693
Epoch 6, Loss: 1.298527542739878
Epoch 7, Loss: 1.2614415257483187
Epoch 8, Loss: 1.2275689979800788
Epoch 9, Loss: 1.1952555586614877
Epoch 10, Loss: 1.166818328525709


In [24]:
# Validation loop
val_loss = 0.0
correct = 0
total = 0

# Switch the model to evaluation mode
model.eval()

with torch.no_grad():  # No need to track gradients during validation
    for images, labels in valloader:
        outputs = model(images)
        loss = criterion(outputs, labels)  # Compute the loss on the validation set
        val_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)  # Get the predicted class with the highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate average loss and accuracy
avg_val_loss = val_loss / len(valloader)
accuracy = 100 * correct / total

print(f"Validation Loss: {avg_val_loss}, Validation Accuracy: {accuracy}%")

Validation Loss: 1.359574123173003, Validation Accuracy: 52.42%
